In [14]:
# 1. 导入依赖并加载电路类
import sympy as sp
from build_circuit_graph_rebuild import Circuit

In [15]:
# 2. 创建空电路并逐步添加元件
circuit = Circuit()

jj1 = circuit.add_component(0, 2, 'JJ', 'EJ1')
jj2 = circuit.add_component(1, 2, 'JJ', 'EJ2')
L = circuit.add_component(0, 1, 'L', 'L')
C1 = circuit.add_component(0, 2, 'C', 'C1')
C2 = circuit.add_component(1, 2, 'C', 'C2')

In [21]:
# 2.1 添加互感示例块（独立示例，不影响主流程）
circuit_mutual = Circuit()

L1_m = circuit_mutual.add_component(0, 1, 'L', 'L1_m')
L2_m = circuit_mutual.add_component(1, 2, 'L', 'L2_m')
C_m = circuit_mutual.add_component(0, 2, 'C', 'C_m')

mid_m = circuit_mutual.add_mutual(L1_m, L2_m, 'M12_m')

print(f'已添加互感: mutual_id={mid_m}, 关联电感元件ID=({C_m}, {L2_m})')
circuit_mutual.print_edges()

已添加互感: mutual_id=0, 关联电感元件ID=(2, 1)
树枝数: 2, 连支数: 1, 总支路数: 3
  Key 0 [树枝]: L (0->1) 值: L1_m [元件ID: 0]
  Key 1 [树枝]: L (1->2) 值: L2_m [元件ID: 1]
  Key 2 [连支]: C (0->2) 值: C_m [元件ID: 2]


In [3]:
# 3. 打印元件列表
circuit.print_components()

共 5 个元件:
  元件 0: JJ (0-2) 值: EJ1
  元件 1: JJ (1-2) 值: EJ2
  元件 2: L (0-1) 值: L
  元件 3: C (0-2) 值: C1
  元件 4: C (1-2) 值: C2


In [4]:
# 4. 打印重排后的边信息
circuit.print_edges()

树枝数: 2, 连支数: 3, 总支路数: 5
  Key 0 [树枝]: JJ (0->2) 值: EJ1 [元件ID: 0]
  Key 1 [树枝]: JJ (1->2) 值: EJ2 [元件ID: 1]
  Key 2 [连支]: L (0->1) 值: L [元件ID: 2]
  Key 3 [连支]: C (0->2) 值: C1 [元件ID: 3]
  Key 4 [连支]: C (1->2) 值: C2 [元件ID: 4]


In [5]:
# 5. 打印基本回路（无外磁通）
circuit.print_loops()

电路共有 3 个基本回路 (树枝数: 2, 总支路数: 5)

回路 0: (由连支 Key 2 闭合)
  连支: L (0->1), 值: L [元件ID: 2]
  回路路径: 0 -> 2 -> 1 -> 0
  包含树枝:
    Key 0: JJ (0->2), 值: EJ1 [元件ID: 0]
    Key 1: JJ (1->2), 值: EJ2 [元件ID: 1]
  外磁通: 0

回路 1: (由连支 Key 3 闭合)
  连支: C (0->2), 值: C1 [元件ID: 3]
  回路路径: 0 -> 2 -> 0
  包含树枝:
    Key 0: JJ (0->2), 值: EJ1 [元件ID: 0]
  外磁通: 0

回路 2: (由连支 Key 4 闭合)
  连支: C (1->2), 值: C2 [元件ID: 4]
  回路路径: 1 -> 2 -> 1
  包含树枝:
    Key 1: JJ (1->2), 值: EJ2 [元件ID: 1]
  外磁通: 0



In [9]:
# 6. 添加物理外磁通并验证回路磁通来源
fid = circuit.add_physical_flux(comp_loop=[3,2,1], flux='Phi_e')


In [10]:
circuit.print_physical_fluxes()
print()
circuit.print_loops()

共 1 个物理磁通:
  磁通 0: Φ = -Phi_e
    回路元件: [1 -> 2 -> 3] = JJ(2→1) -> L(1→0) -> C(0→2)

电路共有 3 个基本回路 (树枝数: 2, 总支路数: 5)

回路 0: (由连支 Key 2 闭合)
  连支: L (0->1), 值: L [元件ID: 2]
  回路路径: 0 -> 2 -> 1 -> 0
  包含树枝:
    Key 0: JJ (0->2), 值: EJ1 [元件ID: 0]
    Key 1: JJ (1->2), 值: EJ2 [元件ID: 1]
  外磁通: Phi_e
  磁通来源: -1 × -Phi_e (磁通ID 0)

回路 1: (由连支 Key 3 闭合)
  连支: C (0->2), 值: C1 [元件ID: 3]
  回路路径: 0 -> 2 -> 0
  包含树枝:
    Key 0: JJ (0->2), 值: EJ1 [元件ID: 0]
  外磁通: -Phi_e
  磁通来源: +1 × -Phi_e (磁通ID 0)

回路 2: (由连支 Key 4 闭合)
  连支: C (1->2), 值: C2 [元件ID: 4]
  回路路径: 1 -> 2 -> 1
  包含树枝:
    Key 1: JJ (1->2), 值: EJ2 [元件ID: 1]
  外磁通: 0



In [11]:
# 7. 显示基本割集矩阵 F_C
sp.pprint(circuit.F_C)

⎡1  0  1   1  0⎤
⎢              ⎥
⎣0  1  -1  0  1⎦


In [12]:
# 8. 计算并展示哈密顿量及关键中间量
H, info = circuit.hamiltonian()

print('\n广义坐标 (树枝磁通):')
sp.pprint(info['phi_t'])

print('\n外磁通变量:')
if info['ext_fluxes']:
    sp.pprint(info['ext_fluxes'])
else:
    print('无')

print('\n全支路磁通 Phi_vec:')
sp.pprint(info['Phi_vec'])

print('\n哈密顿量 H:')
sp.pprint(sp.expand(H))


广义坐标 (树枝磁通):
⎡Φₜ ₀⎤
⎢    ⎥
⎣Φₜ ₁⎦

外磁通变量:
[Φₑ]

全支路磁通 Phi_vec:
⎡      Φₜ ₀      ⎤
⎢                ⎥
⎢      Φₜ ₁      ⎥
⎢                ⎥
⎢Φₑ + Φₜ ₀ - Φₜ ₁⎥
⎢                ⎥
⎢   -Φₑ + Φₜ ₀   ⎥
⎢                ⎥
⎣      Φₜ ₁      ⎦

哈密顿量 H:
                                   2                           2               ↪
                                 Φₑ    Φₑ⋅Φₜ ₀   Φₑ⋅Φₜ ₁   Φₜ ₀    Φₜ ₀⋅Φₜ ₁   ↪
-EJ₁⋅cos(Φₜ ₀) - EJ₂⋅cos(Φₜ ₁) + ─── + ─────── - ─────── + ───── - ───────── + ↪
                                 2⋅L      L         L       2⋅L        L       ↪

↪      2       2       2
↪  Φₜ ₁    Qₜ ₁    Qₜ ₀ 
↪  ───── + ───── + ─────
↪   2⋅L    2⋅C₂    2⋅C₁ 
